In [1]:
import os
import sys

# Add your module's directory (absolute path or relative path)
module_path = os.path.abspath(os.path.join('../src')) 
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [2]:
from frozen_lake import FrozenLake
from policy_metrics import probability_success, mean_return, play_episode, std_return
from display import print_policy, print_transitions, print_state_value_function, print_environment

from models import Environment
from dp import policy_improvement, policy_evaluation, policy_iteration
import random
import numpy as np

In [3]:
def path_policy(s):
    """Hand-picked safe path on the classic 4x4 board:
       0 -> 4 -> 8 -> 9 -> 10 -> 14 -> 15  (avoids holes 5, 7, 11, 12)
    """
    LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

    path = {0: DOWN, 4: DOWN, 8: RIGHT, 9: RIGHT, 10: RIGHT, 14: DOWN}
    if s in path:
        return path[s]
    row, col = divmod(s, 4)                       # fallback: head down-right
    if row < 3:
        return DOWN
    if col < 3:
        return RIGHT
    return LEFT

In [4]:
fl = FrozenLake(n=4, slippery=True, seed=0)
print_environment(fl)
print_transitions(fl.P, fl.actions, states=[0, 5, 15])
play_episode(fl, path_policy, delay=0.5)

Environment:
Legend: S=start  F=frozen  H=hole  G=goal  A=agent
| SA    |F     |F     |F     |
|   0   |  1   |  2   |  3   |
| F     |H     |F     |H     |
|   4   |  5   |  6   |  7   |
| F     |F     |F     |H     |
|   8   |  9   | 10   | 11   |
| H     |F     |F     |G     |
|  12   | 13   | 14   | 15   |
Transition matrix P:
  P(s'= 0 | s=0, a=LEFT ) = 0.67  reward=0.0  terminated=False
  P(s'= 4 | s=0, a=LEFT ) = 0.33  reward=0.0  terminated=False
  P(s'= 0 | s=0, a=DOWN ) = 0.33  reward=0.0  terminated=False
  P(s'= 4 | s=0, a=DOWN ) = 0.33  reward=0.0  terminated=False
  P(s'= 1 | s=0, a=DOWN ) = 0.33  reward=0.0  terminated=False
  P(s'= 4 | s=0, a=RIGHT) = 0.33  reward=0.0  terminated=False
  P(s'= 1 | s=0, a=RIGHT) = 0.33  reward=0.0  terminated=False
  P(s'= 0 | s=0, a=RIGHT) = 0.33  reward=0.0  terminated=False
  P(s'= 1 | s=0, a=UP   ) = 0.33  reward=0.0  terminated=False
  P(s'= 0 | s=0, a=UP   ) = 0.67  reward=0.0  terminated=False
  P(s'= 5 | s=5, a=LEFT ) = 1.00  rew

(0.0, 5)

In [5]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

In [6]:
random_policy = lambda s: random.choice([0, 1, 2, 3])
print("Random Policy")
print_policy(random_policy, fl)
V = policy_evaluation(env=fl, pi=random_policy)
print_state_value_function(V, fl)
print(f"Probability of success: {probability_success(fl, random_policy)}, Mean Return: {mean_return(fl, random_policy)}, Std Return: {std_return(fl, random_policy):0.2f}")


Random Policy
Policy:  (* = agent's current state)
|  00     ^ |  01     v |  02     v |  03     > |
|  04     v | *   H     |  06     > |     H     |
|  08     ^ |  09     v |  10     v |     H     |
|     H     |  13     < |  14     < |     G     |
State-value function:
|  00 0.011 |  01 0.007 |  02 0.011 |  03 0.011 |
|  04 0.009 | *   H     |  06 0.010 |     H     |
|  08 0.032 |  09 0.091 |  10 0.033 |     H     |
|     H     |  13 0.232 |  14 0.506 |     G     |
Probability of success: 0.03, Mean Return: 0.01, Std Return: 0.14


In [7]:
go_get_pi = lambda s: {
    0:RIGHT, 1:RIGHT, 2:DOWN, 3:LEFT,
    4:DOWN, 5:LEFT, 6:DOWN, 7:LEFT,
    8:RIGHT, 9:RIGHT, 10:DOWN, 11:LEFT,
    12:LEFT, 13:RIGHT, 14:RIGHT, 15:LEFT
}[s]
print("Go Get Pi")
print_policy(go_get_pi, fl)
V = policy_evaluation(env=fl, pi=go_get_pi)
print_state_value_function(V, fl)
print(f"Probability of success: {probability_success(fl, go_get_pi)}, Mean Return: {mean_return(fl, go_get_pi)}, Std Return: {std_return(fl, go_get_pi):0.2f}")


Go Get Pi
Policy:  (* = agent's current state)
|  00     > |  01     > |  02     v |  03     < |
|  04     v | *   H     |  06     v |     H     |
|  08     > |  09     > |  10     v |     H     |
|     H     |  13     > |  14     > |     G     |
State-value function:
|  00 0.034 |  01 0.023 |  02 0.047 |  03 0.023 |
|  04 0.046 | *   H     |  06 0.096 |     H     |
|  08 0.094 |  09 0.239 |  10 0.290 |     H     |
|     H     |  13 0.433 |  14 0.640 |     G     |
Probability of success: 0.05, Mean Return: 0.07, Std Return: 0.17


In [8]:
careful_pi = lambda s: {
    0:LEFT, 1:UP, 2:UP, 3:UP,
    4:LEFT, 5:LEFT, 6:UP, 7:LEFT,
    8:UP, 9:DOWN, 10:LEFT, 11:LEFT,
    12:LEFT, 13:RIGHT, 14:RIGHT, 15:LEFT
}[s]
print("Careful Pi")
print_policy(careful_pi, fl)
V = policy_evaluation(env=fl, pi=careful_pi)
print_state_value_function(V, fl)
print(f"Probability of success: {probability_success(fl, careful_pi)}, Mean Return: {mean_return(fl, careful_pi)}, Std Return: {std_return(fl, careful_pi):0.2f}")

Careful Pi
Policy:  (* = agent's current state)
|  00     < |  01     ^ |  02     ^ |  03     ^ |
|  04     < | *   H     |  06     ^ |     H     |
|  08     ^ |  09     v |  10     < |     H     |
|     H     |  13     > |  14     > |     G     |
State-value function:
|  00 0.408 |  01 0.375 |  02 0.354 |  03 0.344 |
|  04 0.420 | *   H     |  06 0.117 |     H     |
|  08 0.445 |  09 0.484 |  10 0.433 |     H     |
|     H     |  13 0.588 |  14 0.711 |     G     |
Probability of success: 0.65, Mean Return: 0.61, Std Return: 0.50


In [9]:
fl.reset()
improved_policy, improved_V = policy_iteration(env=fl,
                                   init_policy=careful_pi)

print("Careful Pi Plus")
print_policy(improved_policy, fl)
print_state_value_function(improved_V, fl)
print(f"Probability of success: {probability_success(fl, improved_policy)}, Mean Return: {mean_return(fl, improved_policy)}, Std Return: {std_return(fl, improved_policy):0.2f}")

Careful Pi Plus
Policy:  (* = agent's current state)
| *00     < |  01     ^ |  02     ^ |  03     ^ |
|  04     < |     H     |  06     < |     H     |
|  08     ^ |  09     v |  10     < |     H     |
|     H     |  13     > |  14     v |     G     |
State-value function:
| *00 0.542 |  01 0.499 |  02 0.471 |  03 0.457 |
|  04 0.558 |     H     |  06 0.358 |     H     |
|  08 0.592 |  09 0.643 |  10 0.615 |     H     |
|     H     |  13 0.742 |  14 0.863 |     G     |
Probability of success: 0.76, Mean Return: 0.81, Std Return: 0.32


Adversarial Pi

In [10]:
adversarial_pi = lambda s: {
    0:UP, 1:UP, 2:UP, 3:UP,
    4:UP, 5:LEFT, 6:UP, 7:LEFT,
    8:LEFT, 9:LEFT, 10:LEFT, 11:LEFT,
    12:LEFT, 13:LEFT, 14:LEFT, 15:LEFT
}[s]
print("Adversarial Pi")
print_policy(adversarial_pi, fl)
V = policy_evaluation(env=fl, pi=adversarial_pi)
print_state_value_function(V, fl)
print(f"Probability of success: {probability_success(fl, adversarial_pi)}, Mean Return: {mean_return(fl, adversarial_pi)}, Std Return: {std_return(fl, adversarial_pi):0.2f}")

Adversarial Pi
Policy:  (* = agent's current state)
|  00     ^ |  01     ^ |  02     ^ |  03     ^ |
|  04     ^ |     H     |  06     ^ |     H     |
|  08     < |  09     < |  10     < |     H     |
|     H     |  13     < |  14     < | *   G     |
State-value function:
|  00 0.000 |  01 0.000 |  02 0.000 |  03 0.000 |
|  04 0.000 |     H     |  06 0.000 |     H     |
|  08 0.000 |  09 0.000 |  10 0.000 |     H     |
|     H     |  13 0.000 |  14 0.000 | *   G     |
Probability of success: 0.0, Mean Return: 0.0, Std Return: 0.00


In [11]:
fl.reset()
improved_adversarial_policy, improved_adversarial_V = policy_iteration(env=fl,
                                   init_policy=adversarial_pi)

print("Improved Adversarial Policy")
print_policy(improved_adversarial_policy, fl)
print_state_value_function(improved_adversarial_V, fl)
print(f"Probability of success: {probability_success(fl, improved_adversarial_policy)}, Mean Return: {mean_return(fl, improved_adversarial_policy)}, Std Return: {std_return(fl, improved_adversarial_policy):0.2f}")

Improved Adversarial Policy
Policy:  (* = agent's current state)
| *00     < |  01     ^ |  02     ^ |  03     ^ |
|  04     < |     H     |  06     < |     H     |
|  08     ^ |  09     v |  10     < |     H     |
|     H     |  13     > |  14     v |     G     |
State-value function:
| *00 0.542 |  01 0.499 |  02 0.471 |  03 0.457 |
|  04 0.558 |     H     |  06 0.358 |     H     |
|  08 0.592 |  09 0.643 |  10 0.615 |     H     |
|     H     |  13 0.742 |  14 0.863 |     G     |
Probability of success: 0.78, Mean Return: 0.76, Std Return: 0.35
